In [17]:
import osmnx as ox
import networkx as nx
import os

# Nama file untuk menyimpan data peta
graph_filename = "sidoarjo_drive.graphml"

# 1. Cek apakah file peta sudah ada di folder komputer
if os.path.exists(graph_filename):
    print("File peta ditemukan. Memuat dari penyimpanan lokal... (Cepat)")
    # Load graph dari file lokal (sangat cepat)
    G = ox.load_graphml(graph_filename)
else:
    print("File peta tidak ditemukan. Mendownload dari OpenStreetMap... (Lama, hanya sekali)")
    # Download graph (hanya jalan jika file belum ada)
    place_name = "Sidoarjo, East Java, Indonesia"
    G = ox.graph_from_place(place_name, network_type='drive')
    # Simpan graph ke file agar run berikutnya tidak perlu download lagi
    ox.save_graphml(G, graph_filename)

print("Peta siap digunakan.")

# 2. Koordinat Awal (Rumah) & Tujuan (Stadion)
origin_point = (-7.43716, 112.62222) 
destination_point = (-7.436203216406945, 112.61925386020168) #gor

# 3. Cari titik (node) jalan terdekat
# Tips: Gunakan lat/lon langsung tanpa perlu tukar posisi di versi OSMnx terbaru,
# tapi pastikan urutan argumen sesuai dokumentasi versi OSMnx yang Anda pakai.
# Biasanya: nearest_nodes(G, X, Y) dimana X=Longitude, Y=Latitude
orig_node = ox.distance.nearest_nodes(G, origin_point[1], origin_point[0])
dest_node = ox.distance.nearest_nodes(G, destination_point[1], destination_point[0])

# 4. Hitung rute terpendek (Dijkstra)
try:
    route = nx.shortest_path(G, orig_node, dest_node, weight='length')
    
    # 5. Hitung total jarak
    route_length = nx.path_weight(G, route, weight='length')
    print(f"Jarak tempuh via jalan raya: {route_length / 1000:.2f} km")
    
except nx.NetworkXNoPath:
    print("Tidak ditemukan jalan yang menghubungkan kedua titik ini.")

File peta ditemukan. Memuat dari penyimpanan lokal... (Cepat)
Peta siap digunakan.
Jarak tempuh via jalan raya: 0.37 km


In [15]:
import osmnx as ox
import os
import ast  # Library untuk mengubah string "['a', 'b']" kembali menjadi list ['a', 'b']

# --- KONFIGURASI ---
# Disarankan pakai .graphml agar atribut lengkap
graph_filename = "sidoarjo_all_network.graphml" 

# 1. LOAD / DOWNLOAD GRAPH
if os.path.exists(graph_filename):
    print("File peta ditemukan. Memuat dari local storage...")
    # PENTING: Saat load, data list sering berubah jadi string
    G = ox.load_graphml(graph_filename)
else:
    print("File peta belum ada. Mendownload dari OSM (Proses ini butuh waktu)...")
    place_name = "Kabupaten Sidoarjo, Jawa Timur, Indonesia"
    # Menggunakan 'all' mengambil semua jalur (mobil, motor, jalan kaki)
    G = ox.graph_from_place(place_name, network_type='all')
    ox.save_graphml(G, graph_filename)

print("Peta siap digunakan.")

# --- FUNGSI PEMBERSIH (INTI PERBAIKAN) ---
def clean_osm_tag(tag_value):
    """
    Membersihkan tag OSM yang tidak konsisten (bisa berupa list, string, 
    atau string yang menyerupai list akibat save/load GraphML).
    """
    # 1. Jika data kosong/None
    if tag_value is None:
        return "unknown"
        
    # 2. Jika tipe aslinya sudah List (biasanya saat baru download)
    if isinstance(tag_value, list):
        return tag_value[0] # Ambil yang pertama (biasanya paling relevan)

    # 3. Jika tipe String, tapi bentuknya seperti list "['a', 'b']" (akibat load GraphML)
    if isinstance(tag_value, str) and tag_value.startswith('[') and tag_value.endswith(']'):
        try:
            # Ubah string kembali jadi list sungguhan
            actual_list = ast.literal_eval(tag_value)
            if isinstance(actual_list, list) and len(actual_list) > 0:
                return actual_list[0]
        except (ValueError, SyntaxError):
            pass # Jika gagal parsing, anggap string biasa
            
    # 4. Jika String biasa, kembalikan langsung
    return tag_value

# --- FUNGSI UTAMA ---
def get_road_type(lat, lon, graph):
    # Cari edge terdekat
    u, v, key = ox.distance.nearest_edges(graph, lon, lat)
    
    # Ambil data edge
    edge_data = graph.get_edge_data(u, v, key)
    
    # Ambil tag highway, default 'unknown' jika tidak ada
    raw_road_type = edge_data.get('highway', 'unknown')
    
    # Bersihkan datanya
    clean_type = clean_osm_tag(raw_road_type)
    
    return clean_type

# --- CONTOH PENGUJIAN ---
# Koordinat A (Jalan Raya Pahlawan)
coords_A = (-7.450540049327813, 112.70757213533062) 
tipe_A = get_road_type(coords_A[0], coords_A[1], G)
print(f"Lokasi A ({coords_A}): {tipe_A}")

# Koordinat B (Masuk Gang/Perumahan)
coords_B = (-7.428255347328013, 112.70326253476101) 
tipe_B = get_road_type(coords_B[0], coords_B[1], G)
print(f"Lokasi B ({coords_B}): {tipe_B}")

# Koordinat C (Jalur Pedestrian / Footway - tes network='all')
coords_C = (-7.4480, 112.7180) # Sekitar GOR Delta (mungkin kena track lari/jalan setapak)
tipe_C = get_road_type(coords_C[0], coords_C[1], G)
print(f"Lokasi C ({coords_C}): {tipe_C}")

File peta ditemukan. Memuat dari local storage...
Peta siap digunakan.
Lokasi A ((-7.450540049327813, 112.70757213533062)): residential
Lokasi B ((-7.428255347328013, 112.70326253476101)): residential
Lokasi C ((-7.448, 112.718)): trunk


In [ ]:
origin_point = (-7.43716, 112.62222) 
destination_point = (-7.436203216406945, 112.61925386020168) #gor

In [18]:
import networkx as nx
import osmnx as ox

# --- 1. SETUP (Load Peta) ---
# Pastikan graph G sudah ada di memori (load dari file graphml Anda)
graph_filename = "sidoarjo_drive.graphml"
if 'G' not in locals():
    G = ox.load_graphml(graph_filename)

# --- 2. SETUP TITIK UJI (Ambil acak dari data Anda) ---
# Rumah (Origin)
orig_point = (-7.43716, 112.62222)
orig_node = ox.distance.nearest_nodes(G, orig_point[1], orig_point[0])

# Tujuan (Destination)
dest_point = (-7.436203216406945, 112.61925386020168)
dest_node = ox.distance.nearest_nodes(G, dest_point[1], dest_point[0])

# =========================================================
# CARA 1 (METODE ANDA)
# =========================================================
print("--- METODE 1 (Kode Referensi Anda) ---")
try:
    # Langkah A: Cari rutenya (list node)
    route_list = nx.shortest_path(G, orig_node, dest_node, weight='length')
    
    # Langkah B: Hitung panjang rutenya
    dist_1 = nx.path_weight(G, route_list, weight='length')
    print(f"Hasil Jarak: {dist_1} meter")
except:
    print("Tidak ada rute")

# =========================================================
# CARA 2 (METODE SAYA)
# =========================================================
print("\n--- METODE 2 (Kode Saya) ---")
try:
    # Langkah: Langsung minta panjangnya (Dijkstra juga)
    dist_2 = nx.shortest_path_length(G, source=orig_node, target=dest_node, weight='length')
    print(f"Hasil Jarak: {dist_2} meter")
except:
    print("Tidak ada rute")

# =========================================================
# BUKTI PERBANDINGAN
# =========================================================
print("\n--- KESIMPULAN ---")
if dist_1 == dist_2:
    print("✅ TERBUKTI: Kedua algoritma menghasilkan angka yang SAMA PERSIS.")
else:
    print("❌ BEDA.")

--- METODE 1 (Kode Referensi Anda) ---
Hasil Jarak: 367.2501279300583 meter

--- METODE 2 (Kode Saya) ---
Hasil Jarak: 367.2501279300583 meter

--- KESIMPULAN ---
✅ TERBUKTI: Kedua algoritma menghasilkan angka yang SAMA PERSIS.
